In [1]:
# import sys
# sys.path.append("/Users/jong/Documents/ovgu/spine/spine_interaction/src/")

import lightning as pl
import torch
import numpy as np
import pandas as pd

from gait_ml.model.litmodel import LitSeq2Seq
from gait_ml.plotting import plot_xyz

from gait_ml.data.dataset import GaitDataset
from gait_ml.data.datamodule import GaitDataModule
from glob import glob

from scipy.signal import find_peaks, peak_widths, butter, sosfiltfilt

import plotly.io as pio
pio.renderers.default = 'notebook'

%load_ext autoreload
%autoreload 2

### 1. Checks

In [18]:
# # Alignment check
# for i in all_files:
#     print(f"Checking {i}")
#     acc = pd.read_excel(i, sheet_name="Linear Accelerometer")
#     gyr = pd.read_excel(i, sheet_name="Gyroscope")
#     assert (gyr.iloc[:, 0] - acc.iloc[:, 0]).mean() == 0

### Check training data

In [114]:
# Visualize data
all_files = glob(
    "/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_*_T1/*_1_2mW_IPhone.xls",
    recursive=True,
)
all_files = np.sort(all_files)
all_idx = np.arange(len(all_files)).tolist()
print(f"Processing: {len(all_files)} samples")
window_size = 256
step_size = 64
batch_size = 1
expand_labels=0

# Initialize the GaitDataModule
data_module = GaitDataModule(all_files,
                             batch_size=batch_size,
                             window_size=window_size,
                             step_size=step_size,
                             train_idx=all_idx, 
                             val_idx=all_idx[-2:],
                             test_idx=all_idx[-2:],
                             expand_labels=expand_labels,
                             acc_sheet_name="Linear Accelerometer"
                            )

data_module.setup(stage="train")
data_module.setup(stage="test")

# # # # Access the dataloaders
# train_loader = data_module.train_dataloader()
# val_loader = data_module.val_dataloader()

# # Plot raw data
# for i in train_loader:
#     print(i[0].shape, i[1].shape)
#     fig = plot_xyz(i[0][0, :, 0], i[0][0, :, 1], i[0][0, :, 2], labels=i[1][0, :])
#     fig.show()
#     fig = plot_xyz(i[0][0, :, 3], i[0][0, :, 4], i[0][0, :, 5], labels=i[1][0, :])
#     fig.show()
#     break

Processing: 46 samples
/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_01_T1/01_1_2mW_IPhone.xls
cropped_array shape: (210, 256, 7)
/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_02_T1/02_1_2mW_IPhone.xls
cropped_array shape: (196, 256, 7)
/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_03_T1/03_1_2mW_IPhone.xls
cropped_array shape: (194, 256, 7)
/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_04_T1/04_1_2mW_IPhone.xls
cropped_array shape: (196, 256, 7)
/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_05_T1/05_1_2mW_IPhone.xls
Skipping: 12627:12723 | index 0 is out of bounds for axis 0 with size 0
cropped_array shape: (194, 256, 7)
/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin

### All dataset

In [125]:
for k, (data_x, data_y) in enumerate(zip(data_module.train_dataset.raw_x, data_module.train_dataset.raw_y)):
    if k > 35:
        print(data_module.all_files[k])
        print(data_x.shape, data_y.shape)
        fig = plot_xyz(data_x[:, 2],data_x[:, 2], data_x[:, 2], data_y)
        fig.show()
    

/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_39_T1/39_1_2mW_IPhone.xls
(12761, 6) (12761,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_40_T1/40_1_2mW_IPhone.xls
(13419, 6) (13419,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_41_T1/41_1_2mW_IPhone.xls
(12752, 6) (12752,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_42_T1/42_1_2mW_IPhone.xls
(13295, 6) (13295,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_43_T1/43_1_2mW_IPhone.xls
(13557, 6) (13557,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_44_T1/44_1_2mW_IPhone.xls
(12715, 6) (12715,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_45_T1/45_1_2mW_IPhone.xls
(12834, 6) (12834,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_46_T1/46_1_2mW_IPhone.xls
(12875, 6) (12875,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_47_T1/47_1_2mW_IPhone.xls
(12826, 6) (12826,)


/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_48_T1/48_1_2mW_IPhone.xls
(12875, 6) (12875,)
